# Bibliotecas

In [41]:
import os
import pandas as pd
import numpy as np

import subprocess
import shutil
import requests
import zipfile
import time
import psutil
from selenium import webdriver
from selenium.webdriver.edge.service import Service as EdgeService
from selenium.webdriver.edge.options import Options as EdgeOptions
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import WebDriverException



from datetime import datetime, timedelta

In [42]:
# Obtém o nome do usuário do sistema
usuario = os.getlogin()
print(usuario)

E713611


In [43]:
#caminhos
#caminho_downloads = f'C:\\Users\\{usuario}\\Downloads\\'
caminho_base = (
    fr"C:\Users\{usuario}\EDP"
    r"\O365_Planejamento_Manutencao_EDP_Brasil - Documentos"
    r"\PLANO MANUTENÇÃO"
    r"\Pacote de notas"
    r"\ES"
    r"\Base para Geração de Pacotes"
)
caminho_aux = (
    fr"C:\Users\{usuario}\EDP"
    r"\O365_Planejamento_Manutencao_EDP_Brasil - Documentos"
    r"\PLANO MANUTENÇÃO"
    r"\Pacote de notas"
    r"\ES"
    r"\Auxiliares" 
    )

# email_envio = 'danilop.vilela@edp.com'

# Login_SAP = '709496'
# Senha_SAP = '@Edp2025'

In [44]:
#ano atual
ano = datetime.now().year
# Data de hoje
hoje = pd.Timestamp.today().normalize()
print(ano,hoje)

2026 2026-05-26 00:00:00


# Baixa base de ocorrencias

In [45]:
arquivo1 = os.path.join(caminho_base, "Base_Ocorrencias_ES.csv")
arquivo2 = os.path.join(caminho_base, "Base_Indicadores_Movel_ES.csv")

# Remove se já existir
if os.path.exists(arquivo1):
    os.remove(arquivo1)

# Remove se já existir
if os.path.exists(arquivo2):
    os.remove(arquivo2)


In [46]:
#Sempre instala edgedriver da versão atual
caminho_downloads = caminho_base

arquivo_csv = os.path.join(caminho_base, "Base_Indicadores_Movel_ES.csv")
arquivo_ocorrencias = os.path.join(caminho_base, "Base_Ocorrencias_ES.csv")

for proc in psutil.process_iter(['name']):
    if proc.info['name'] == 'msedgedriver.exe':
        try:
            proc.kill()
        except Exception:
            pass
def acessar():
    """Acessa o site do Coffee"""
    global navegador

    # Determina a versão do Microsoft Edge instalada
    try:
        result = subprocess.run(["reg", "query", "HKEY_CURRENT_USER\\Software\\Microsoft\\Edge\\BLBeacon", "/v", "version"], capture_output=True, text=True, shell=True)
        edge_version = None
        for line in result.stdout.splitlines():
            if "version" in line:
                edge_version = line.split()[-1]
                break
        if not edge_version:
            raise Exception("Não foi possível determinar a versão do Microsoft Edge.")
    except Exception as e:
        print(f"Erro ao obter a versão do Microsoft Edge: {e}")
        os._exit(1)

    # Monta o link de download do EdgeDriver
    edgedriver_url = f"https://msedgedriver.microsoft.com/{edge_version}/edgedriver_win64.zip"

    # Caminho para salvar o arquivo baixado e descompactar
    download_path = "C:\\Selenium\\edgedriver_win64.zip"
    extract_path = "C:\\Selenium\\"

    # Cria a pasta C:\Selenium se ela não existir
    if not os.path.exists(extract_path):
        os.makedirs(extract_path)

    # Faz o download do EdgeDriver
    try:
        response = requests.get(edgedriver_url, stream=True, verify=False)
        if response.status_code == 200:
            with open(download_path, "wb") as file:
                shutil.copyfileobj(response.raw, file)
        else:
            raise Exception(f"Falha no download do EdgeDriver. Status code: {response.status_code}")
    except Exception as e:
        print(f"Erro ao baixar o EdgeDriver: {e}")
        os._exit(1)

    # Descompacta o arquivo baixado
    try:
        with zipfile.ZipFile(download_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        os.remove(download_path)  # Remove o arquivo zip após descompactar
    except Exception as e:
        print(f"Erro ao descompactar o EdgeDriver: {e}")
        os._exit(1)

    # Configurando opções do navegador
    opt = EdgeOptions()
    opt.add_argument("--start-maximized") #Abre em tela cheia
    #opt.add_argument("--headless")  # Executa o navegador em modo invisível

    prefs = {
        "download.default_directory": caminho_downloads,   # pasta para downloads
        "download.prompt_for_download": False,        # não perguntar
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True                  # evita bloqueio
    }
    opt.add_experimental_option("prefs", prefs)

    # Configurando o serviço com o caminho do executável msedgedriver.exe
    service = EdgeService("C:\\Selenium\\msedgedriver.exe")

    navegador = webdriver.Edge(service=service, options=opt)

    #link do site que deseja abrir
    navegador.get("https://adb-2199566635981619.19.azuredatabricks.net/ajax-api/2.0/fs/files/Volumes/poseidon_uc/group_uc/ddpm/Danilo/Base_Ocorrencias_ES.csv")

    count = 0

    while (
        not os.path.exists(arquivo_csv)
        and not os.path.exists(arquivo_ocorrencias)
        and count <= 300
    ):
        print(f"Aguardando download... {count}s")
        time.sleep(5)
        count += 5
    # espera arquivos temporários sumirem
    while any(
        f.endswith(".crdownload")
        for f in os.listdir(caminho_downloads)
    ):
        time.sleep(2)
    print("Download concluído")
    return navegador
acessar()
time.sleep(1)

c:\Users\E713611\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'msedgedriver.microsoft.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Aguardando download... 0s
Aguardando download... 5s
Aguardando download... 10s
Aguardando download... 15s
Aguardando download... 20s
Aguardando download... 25s
Aguardando download... 30s
Aguardando download... 35s
Download concluído


In [47]:
time.sleep(5)
navegador.get("https://adb-2199566635981619.19.azuredatabricks.net/ajax-api/2.0/fs/files/Volumes/poseidon_uc/group_uc/ddpm/Danilo/Base_Indicadores_Movel_ES.csv")
time.sleep(5)

In [48]:
#fecha navegador
count = 0

while (
    not os.path.exists(arquivo1)
    and not os.path.exists(arquivo2)
    and count <= 300
):
    time.sleep(10)
    count += 10

navegador.quit()

In [49]:
Indicadores = pd.read_csv(
    arquivo_csv,
    sep=";",   # tenta ; em vez de ,
    encoding="utf-8"  # se der erro de acento, usar "latin1"
)

display(Indicadores)

,CONJUNTO_DESC,DEC_CONJUNTO,FEC_CONJUNTO,LIM_DEC,LIM_FEC,DELTA_DEC,DELTA_FEC
0,SANTA MARIA DE JETIBÁ,"19,973973878042433","7,8231178674247035","17,0","10,0","2,9739738780424325","-2,1768821325752965"
1,JOAO NEIVA,"12,938601960750997","5,0517593765082704","11,0","7,0","1,9386019607509972","-1,9482406234917296"
2,ARACRUZ,"10,034622887655523","4,030288123510358","10,0","7,0","0,034622887655523016","-2,969711876489642"
3,LINHARES,"9,86276105311263","4,017429046617595","10,0","6,0","-0,13723894688737026","-1,982570953382405"
4,SAO MATEUS,"8,806356811068262","4,186173146503875","9,0","7,0","-0,1936431889317376","-2,813826853496125"
5,CANIVETE,"10,615768896672137","6,076698889609774","11,0","7,0","-0,38423110332786337","-0,9233011103902262"
6,GUARAPARI,"7,381296267928971","2,879017682533889","8,0","5,0","-0,6187037320710287","-2,120982317466111"
7,CEASA,"5,288598050825166","2,442470713501029","6,0","5,0","-0,7114019491748342","-2,557529286498971"
8,PIUMA,"7,27313309704376","3,078683136184365","8,0","6,0","-0,7268669029562398","-2,921316863815635"
9,IBES,"4,105852240016425","2,1018503505425583","5,0","4,0","-0,894147759983575","-1,8981496494574417"


In [50]:
Ocorrencias = pd.read_csv(
    arquivo_ocorrencias,
    sep=";",
    encoding="utf-8"
)

display(Ocorrencias)

,ORDEM_ID,ANO,MES,DIA,CONJUNTO_ANEEL,CONJUNTO_DESC,CSD,CIRCUITO,CLASSE,CLASSE_DESC,...,TE,TA,DURACAO,QTD_OCO,CI,CHI,DEC_CONJUNTO,FEC_CONJUNTO,DEC_EMPRESA,FEC_EMPRESA
0,10312666-1,2026,3,13,618,MARECHAL FLORIANO,GUARAPARI,MFL04,201.0,201-Desligamento rede Primária,...,"13,0","1623,0","1622,0",1,"1,0","27,033333333333335","0,0009702581772067093","3,589117794846027e-05","1,502749327144452e-05","5,558875439498589e-07"
1,10353540-1,2026,4,11,618,MARECHAL FLORIANO,GUARAPARI,MFL03,201.0,201-Desligamento rede Primária,...,"48,0","244,0","244,0",1,"2,0","8,133333333333333","0,00029101664996899004","7,156147130385e-05","4,510544877518372e-06","1,1091503797176326e-06"
2,10073040-1,2025,9,22,609,BARRA DE SÃO FRANCISCO,NOVA VENÉCIA,SFR06,201.0,201-Desligamento rede Primária,...,"9,0","267,0","264,0",1,"1,0","4,4","0,00010567016498955306","2,4015946588534786e-05","2,471757363309009e-06","5,617630371156839e-07"
3,10138737-1,2025,11,15,612,FRUTEIRAS,CACHOEIRO,FRU02,350.0,350-Serviço Ramal / Medição com reclamação,...,"75,0","173,0","168,0",1,"1,0","2,8","0,00010393081177387624","3,711814706209866e-05","1,5669700619177027e-06","5,596321649706081e-07"
4,10336697-1,2026,3,28,401,AFONSO CLAUDIO,CACHOEIRO,AFC01,350.0,350-Serviço Ramal / Medição com reclamação,...,"5,0","349,0","349,0",1,"1,0","5,816666666666666","0,00026057999581877373","4,479885314935937e-05","3,2334125473083464e-06","5,558875439498589e-07"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151956,10143582-1,2025,11,19,617,JAGUARE,LINHARES,JGR05,201.0,201-Desligamento rede Primária,...,"120,0","476,0","476,0",1,"2,0","15,866666666666667","0,0011433787321947586","0,00014412336960438135","8,879497017533648e-06","1,1192643299412161e-06"
151957,9991561-1,2025,7,1,448,CARAPINA,VITÓRIA,CAR05,244.0,244-Desligamento Programado BT/MT,...,NaN,"0,0","926,0",1,"7,0","54,016666666666666","0,0006040578672899217","7,827963722979546e-05","3,044204364643271e-05","3,944973258716982e-06"
151958,10368531-1,2026,4,25,421,GUACUI,CACHOEIRO,GUI03,201.0,201-Desligamento rede Primária,...,"14,0","231,0","231,0",1,"1,0","3,85","0,00019442480557519442","5,04999495000505e-05","2,1351144809564426e-06","5,545751898588163e-07"
151959,10101491-1,2025,10,30,498,CARIACICA,VITÓRIA,CRC13,244.0,244-Desligamento Programado BT/MT,...,NaN,"0,0","122,0",1,"43,0","87,43333333333334","0,0020912562685865086","0,0010284866894687747","4,9019937640210476e-05","2,410816605256253e-05"


## Cria dinamica por blocos

In [51]:
colunas_numericas = ['CI', 'CHI']
for col in colunas_numericas:
    Ocorrencias[col] = (
        Ocorrencias[col]
            .astype(str)
            .str.replace(',', '.', regex=False)
            .str.strip()
    )

    Ocorrencias[col] = pd.to_numeric(
        Ocorrencias[col],
        errors='coerce'
    ).fillna(0)

# =========================
# Definir tipos finais
# =========================

Ocorrencias['CI'] = Ocorrencias['CI'].round().astype('Int64')
Ocorrencias['CHI'] = Ocorrencias['CHI'].astype(float)

Ocorrencias

,ORDEM_ID,ANO,MES,DIA,CONJUNTO_ANEEL,CONJUNTO_DESC,CSD,CIRCUITO,CLASSE,CLASSE_DESC,...,TE,TA,DURACAO,QTD_OCO,CI,CHI,DEC_CONJUNTO,FEC_CONJUNTO,DEC_EMPRESA,FEC_EMPRESA
0,10312666-1,2026,3,13,618,MARECHAL FLORIANO,GUARAPARI,MFL04,201.0,201-Desligamento rede Primária,...,"13,0","1623,0","1622,0",1,1,27.033333,"0,0009702581772067093","3,589117794846027e-05","1,502749327144452e-05","5,558875439498589e-07"
1,10353540-1,2026,4,11,618,MARECHAL FLORIANO,GUARAPARI,MFL03,201.0,201-Desligamento rede Primária,...,"48,0","244,0","244,0",1,2,8.133333,"0,00029101664996899004","7,156147130385e-05","4,510544877518372e-06","1,1091503797176326e-06"
2,10073040-1,2025,9,22,609,BARRA DE SÃO FRANCISCO,NOVA VENÉCIA,SFR06,201.0,201-Desligamento rede Primária,...,"9,0","267,0","264,0",1,1,4.400000,"0,00010567016498955306","2,4015946588534786e-05","2,471757363309009e-06","5,617630371156839e-07"
3,10138737-1,2025,11,15,612,FRUTEIRAS,CACHOEIRO,FRU02,350.0,350-Serviço Ramal / Medição com reclamação,...,"75,0","173,0","168,0",1,1,2.800000,"0,00010393081177387624","3,711814706209866e-05","1,5669700619177027e-06","5,596321649706081e-07"
4,10336697-1,2026,3,28,401,AFONSO CLAUDIO,CACHOEIRO,AFC01,350.0,350-Serviço Ramal / Medição com reclamação,...,"5,0","349,0","349,0",1,1,5.816667,"0,00026057999581877373","4,479885314935937e-05","3,2334125473083464e-06","5,558875439498589e-07"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151956,10143582-1,2025,11,19,617,JAGUARE,LINHARES,JGR05,201.0,201-Desligamento rede Primária,...,"120,0","476,0","476,0",1,2,15.866667,"0,0011433787321947586","0,00014412336960438135","8,879497017533648e-06","1,1192643299412161e-06"
151957,9991561-1,2025,7,1,448,CARAPINA,VITÓRIA,CAR05,244.0,244-Desligamento Programado BT/MT,...,NaN,"0,0","926,0",1,7,54.016667,"0,0006040578672899217","7,827963722979546e-05","3,044204364643271e-05","3,944973258716982e-06"
151958,10368531-1,2026,4,25,421,GUACUI,CACHOEIRO,GUI03,201.0,201-Desligamento rede Primária,...,"14,0","231,0","231,0",1,1,3.850000,"0,00019442480557519442","5,04999495000505e-05","2,1351144809564426e-06","5,545751898588163e-07"
151959,10101491-1,2025,10,30,498,CARIACICA,VITÓRIA,CRC13,244.0,244-Desligamento Programado BT/MT,...,NaN,"0,0","122,0",1,43,87.433333,"0,0020912562685865086","0,0010284866894687747","4,9019937640210476e-05","2,410816605256253e-05"


In [52]:
causas_gerenciaveis = (11, 20, 130, 180, 181, 182, 200, 322, 323, 324, 327, 329, 491, 492, 820, 880, 890)

Ocorrencias = Ocorrencias[Ocorrencias['TIPO_DESLIGAMENTO'] == 'ACIDENTAL']
print(len(Ocorrencias))
Ocorrencias = Ocorrencias[Ocorrencias['CAUSA'].isin(causas_gerenciaveis)]

print(len(Ocorrencias))

144304
131964


In [53]:
# Cria coluna de data a partir de ANO, MES, DIA
Ocorrencias['DATA'] = pd.to_datetime(Ocorrencias[['ANO', 'MES', 'DIA']].astype(str).agg('-'.join, axis=1), format='%Y-%m-%d')

# Data limite: 90 dias atrás da data atual
data_limite = datetime.now() - timedelta(days=90)

# Filtra por datas nos últimos 90 dias
df_ultimos_90 = Ocorrencias[Ocorrencias['DATA'] >= data_limite]

# Conta distintos ORDEM_ID nos últimos 90 dias por EQ_INTERRUPCAO
qtd_90d = df_ultimos_90.groupby('EQ_INTERRUPCAO')['ORDEM_ID'].nunique().reset_index().rename(columns={'ORDEM_ID': 'QTD_Ocorrencias_90d'})

# Junta com o grouped_df original
grouped_df = Ocorrencias.groupby('EQ_INTERRUPCAO').agg({'ORDEM_ID': 'nunique','CI': 'sum', 'CHI': 'sum'}).reset_index().rename(columns={'ORDEM_ID': 'QTD_Ocorrencias'})
grouped_df = grouped_df.merge(qtd_90d, on='EQ_INTERRUPCAO', how='left').fillna({'QTD_Ocorrencias_90d': 0})

# # =========================
# # 6️⃣ Garantir que tudo vire número
# # =========================

# =========================
# 7⃣️ Definir tipos finais
# =========================

grouped_df['QTD_Ocorrencias'] = grouped_df['QTD_Ocorrencias'].astype('Int64')
grouped_df['QTD_Ocorrencias_90d'] = grouped_df['QTD_Ocorrencias_90d'].astype('Int64')
grouped_df['CI'] = grouped_df['CI'].round().astype('Int64')
grouped_df['CHI'] = grouped_df['CHI'].astype(float)

# =========================
# 8⃣️ Ajuste final na chave
# =========================
grouped_df['EQ_INTERRUPCAO'] = grouped_df['EQ_INTERRUPCAO'].astype(str) + '9'

Dim_Ocorrencias = grouped_df

display(Dim_Ocorrencias)

,EQ_INTERRUPCAO,QTD_Ocorrencias,CI,CHI,QTD_Ocorrencias_90d
0,701BC000028729,1,393,2740.183333,1
1,701BF000002539,1,182,388.266667,0
2,701BF000005769,1,93,323.950000,1
3,701BF000005839,2,420,1837.500000,0
4,701BF000006259,1,106,176.666667,1
...,...,...,...,...,...
87455,Seccionamento Temporário 969459,1,4,20.800000,1
87456,Seccionamento Temporário 969469,1,1350,1608.450000,1
87457,Seccionamento Temporário 969499,1,30,25.000000,1
87458,Seccionamento Temporário 969509,1,8,7.466667,1


# Base de notas

## Base de notas do SQL (Não tem supervisão)

In [54]:
# import pandas as pd
# import pyodbc
# from sqlalchemy import create_engine

# # Lê o arquivo de Acesso ao SQL Server
# df_acessoSQL = pd.read_csv('\\\\fscoc10\\dep\\DDPM\\COFFEE\\Gerador de Notas\\AcessoSQL.csv', sep=';')
# u_sql = df_acessoSQL['Acesso'][0]
# p_sql = df_acessoSQL['Acesso'][1]

# ## SQL Server
# # Configuração da string de conexão
# print('Criando string de conexão...')
# server = 'edpbredb.database.windows.net'
# database = 'ddpm'
# username_ad = u_sql  # Substitua pelo seu usuário
# password = p_sql  # Substitua pela sua senha

# # String de conexão
# conn_str = (
#     f'DRIVER={{ODBC Driver 17 for SQL Server}};'
#     f'SERVER={server};'
#     f'DATABASE={database};'
#     f'Authentication=ActiveDirectoryPassword;'
#     f'UID={username_ad};'
#     f'PWD={password};'
#     f'TrustServerCertificate=yes;'
# )

# # Conexão com o banco de dados
# print('Conectando ao banco de dados...')
# connection_url = f"mssql+pyodbc:///?odbc_connect={conn_str}"
# engine = create_engine(connection_url)

# ## Ler DataFrame do SQL Server
# print('Lendo DataFrame do SQL Server...')

# #df = pd.read_sql('SELECT * from NotasDONR', engine)

# # Imprimir DataFrame
# #print(df)

In [55]:
# notas = pd.read_sql('''SELECT *  FROM NotasDONR
#                  WHERE
#                      EMPRESA = 380 --391 SP-- Mudar filtro da empresa 
#                      AND (local_instalacao IS NULL or local_instalacao LIKE '7%') -- Mudar filtro de local de instalação 
#                      AND [descrição_conjunto] IN (
#                         'BF-CHAVE FUSIVEL <34,5 kV- OPEX',
#                         'MANUT.  EQUIPAMENTOS - OPEX',
#                         'MANUT. CIRC. PRIMARIO - MT - OPEX',
#                         'MANUT. CIRC. SECUNDARIO-BT - OPEX',
#                         'PODA DE ARVORES - OPEX'
#                      )
#                      AND id_sap NOT IN ('99999999', '999999999')
#                      AND Status_SAP <> 'Encerrado' AND Status_SAP <> 'Cancelado'
#                      AND recebido >= '2024-01-01T00:00:00'

#                  ''', engine)

# # Imprimir DataFrame
# display(notas)

In [56]:
# notas['recebido'] = pd.to_datetime(notas['recebido'], errors='coerce')
# # Cria colunas 'ANO', 'MES' e 'DIA' a partir da coluna 'Data'
# notas['ANO'] = notas['recebido'].dt.year
# notas['MES'] = notas['recebido'].dt.month
# notas['DIA'] = notas['recebido'].dt.day

In [57]:
# display(notas['ANO'].unique())

## Le base de Notas

In [58]:
#Baixar tabelas de ids pendentes 

In [59]:
Notas = pd.read_excel(
    fr'{caminho_base}\Notas_Pendentes.xlsx')

display(Notas)

c:\Users\E713611\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Solicitante,id_onr,id_sap,CSD,SUPERVISAO,CJ_NOME,SE_NOME,recebido,alimentador,local_instalacao,...,DATA_PREVISTA_EXECUÇÃO_PROTHEUS,sintoma,data_fabricacao_poste,postes,Status_SAP,latitude,longitude,Croqui,observacoes,IdsDeolhoRede
0,Planejamento,303601.0,NaN,CENTRO,DESU-CR,ALTO LAGE,ALTO LAGE,2026-05-26 18:01:08.000,ALA35,703CF00476950,...,NaT,DETE,NaN,12587506,NaN,-2.033450e+01,-4.038944e+01,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Em atendimento a ordem 10365190-1 a equipe EPV...,1.0
1,Planejamento,303591.0,10000000.0,CENTRO,DESG-MF,CEASA,SANTA ISABEL,2026-05-26 17:41:20.973,SIS02,704ET00200659,...,NaT,DESP,NaN,12445677,NaN,-2.026527e+01,-4.037238e+01,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,"Substituir poste PR1 com TF com risco de cair,...",1.0
2,CSD,303587.0,NaN,LINHARES,DESL-LI,CANIVETE,JUNCADO,2026-05-26 17:28:22.813,JUN05,770BF00005113,...,NaT,DETE,NaN,NaN,NaN,-1.911872e+01,-4.008620e+01,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Carro linha viva vem debaixo poste estrutura N1,1.0
3,CSD,303585.0,NaN,LINHARES,DESL-LI,LINHARES,LINHARES,2026-05-26 17:24:08.837,LIN16,723BF00005113,...,NaT,DETE,NaN,NaN,NaN,-1.911883e+01,-4.008613e+01,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Carro linha viva vem debaixo poste estrutura N1,1.0
4,Planejamento,303584.0,NaN,GUARAPARI,DESG-MT,ITAPEMIRIM,MARATAIZES,2026-05-26 17:21:24.483,MRT03,768ET00055974,...,NaT,DETE,NaN,12255542,NaN,-2.100828e+01,-4.080733e+01,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130186,Planejamento,15.0,12484333.0,ITARANA,DESI-JN,BARRA DO SAHY,BARRA DO SAHY,2018-09-13 15:34:14.830,FUN01,714RL00001114,...,NaT,ARVO,NaN,NaN,NaN,-2.031513e+01,-4.029022e+01,https://coffee.edp.gpti.com.br/olhonarede/pdf/15,NaN,1.0
130187,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
130188,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,...,NaT,NaN,NaN,NaN,NaN,1.055034e+08,5.087045e+09,NaN,NaN,130187.0
130189,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [60]:
#Garante que todas as notas estejam pedentes e sejam de manutenção ou poda
Notas = Notas[Notas['descrição_conjunto'].str.contains('MANUT.|PODA', case=False, na=False)]
Notas = Notas[Notas['Data_encerramento_exec'].isnull()]

In [61]:
# Criando coluna com diferença em dias
Notas['tempo de nota'] = (hoje - Notas['recebido']).dt.days
display(Notas)

,Solicitante,id_onr,id_sap,CSD,SUPERVISAO,CJ_NOME,SE_NOME,recebido,alimentador,local_instalacao,...,sintoma,data_fabricacao_poste,postes,Status_SAP,latitude,longitude,Croqui,observacoes,IdsDeolhoRede,tempo de nota
2,CSD,303587.0,NaN,LINHARES,DESL-LI,CANIVETE,JUNCADO,2026-05-26 17:28:22.813,JUN05,770BF00005113,...,DETE,NaN,NaN,NaN,-19.118719,-40.086204,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Carro linha viva vem debaixo poste estrutura N1,1.0,-1
3,CSD,303585.0,NaN,LINHARES,DESL-LI,LINHARES,LINHARES,2026-05-26 17:24:08.837,LIN16,723BF00005113,...,DETE,NaN,NaN,NaN,-19.118830,-40.086135,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Carro linha viva vem debaixo poste estrutura N1,1.0,-1
6,CSD,303580.0,NaN,CACHOEIRO,DESC-AL,MIMOSO DO SUL,MIMOSO,2026-05-26 17:13:26.047,MSU01,743CF00000078,...,ARVO,NaN,NaN,NaN,-20.916632,-41.539976,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Corte de eucalipto ao lado da BF1020,1.0,-1
8,CSD,303577.0,NaN,CACHOEIRO,DESC-AL,ALEGRE,ALEGRE,2026-05-26 17:01:21.207,ALE05,743CF00000078,...,ARVO,NaN,NaN,NaN,-20.911188,-41.543312,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Corte de palmitos.,1.0,-1
9,Planejamento,303569.0,NaN,CENTRO,DEST-SE,CIVIT,JACARAÍPE,2026-05-26 16:41:53.040,JAC03,707BF00004239,...,ARVO,NaN,NaN,NaN,-20.148183,-40.193021,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Cortar cipó poste concreto rede MT cabo nu 3x0...,1.0,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130182,Planejamento,19.0,12509885.0,CENTRO,DESU-CR,CEASA,VIANA,2018-09-13 17:09:17.547,VIA05,704BF00000333,...,DETE,NaN,NaN,NaN,-20.314862,-40.290219,https://coffee.edp.gpti.com.br/olhonarede/pdf/19,NaN,1.0,2811
130183,Planejamento,18.0,12865479.0,CENTRO,DEST-VI,BENTO FERREIRA,BENTO FERREIRA,2018-09-13 16:56:20.000,BFE06,701BF00004643,...,DETE,NaN,NaN,NaN,-20.314848,-40.290244,https://coffee.edp.gpti.com.br/olhonarede/pdf/18,NaN,1.0,2811
130184,Planejamento,17.0,10000000.0,CENTRO,DESU-CR,CARIACICA,CARIACICA,2018-09-13 15:38:43.150,CRC11,703BF00000023,...,DETE,NaN,NaN,NaN,-20.314846,-40.290246,https://coffee.edp.gpti.com.br/olhonarede/pdf/17,Final circuito,1.0,2811
130185,Planejamento,16.0,10000000.0,CENTRO,DEST-SE,CARAPINA,CARAPINA,2018-09-13 15:36:24.500,CAR06,707CF00000112,...,DETE,NaN,NaN,NaN,-20.314845,-40.290242,https://coffee.edp.gpti.com.br/olhonarede/pdf/16,NaN,1.0,2811


In [62]:
condicoes = [
    Notas['tempo de nota'] <= 365,
    (Notas['tempo de nota'] > 365) & (Notas['tempo de nota'] <= 730),
    Notas['tempo de nota'] > 730
]

categorias = [
    'Até 1 ano',
    'Até 2 anos',
    'Mais de 2 anos'
]

Notas['Classificação Tempo'] = np.select(condicoes, categorias, default='Sem data')
Notas = Notas[Notas['Classificação Tempo'].isin(['Até 1 ano','Até 2 anos'])]

In [63]:
Notas_bk = Notas.copy()
#Notas = Notas_bk

In [64]:
len(Notas)

23106

## Remove Medidas que não são manutenção

In [65]:
Notas = Notas[~Notas['componente_novo'].str.contains('abelha|Aprumar poste|Separador|Placa', case=False, na=False)]

In [66]:
Notas

,Solicitante,id_onr,id_sap,CSD,SUPERVISAO,CJ_NOME,SE_NOME,recebido,alimentador,local_instalacao,...,data_fabricacao_poste,postes,Status_SAP,latitude,longitude,Croqui,observacoes,IdsDeolhoRede,tempo de nota,Classificação Tempo
2,CSD,303587.0,NaN,LINHARES,DESL-LI,CANIVETE,JUNCADO,2026-05-26 17:28:22.813,JUN05,770BF00005113,...,NaN,NaN,NaN,-19.118719,-40.086204,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Carro linha viva vem debaixo poste estrutura N1,1.0,-1,Até 1 ano
3,CSD,303585.0,NaN,LINHARES,DESL-LI,LINHARES,LINHARES,2026-05-26 17:24:08.837,LIN16,723BF00005113,...,NaN,NaN,NaN,-19.118830,-40.086135,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Carro linha viva vem debaixo poste estrutura N1,1.0,-1,Até 1 ano
6,CSD,303580.0,NaN,CACHOEIRO,DESC-AL,MIMOSO DO SUL,MIMOSO,2026-05-26 17:13:26.047,MSU01,743CF00000078,...,NaN,NaN,NaN,-20.916632,-41.539976,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Corte de eucalipto ao lado da BF1020,1.0,-1,Até 1 ano
8,CSD,303577.0,NaN,CACHOEIRO,DESC-AL,ALEGRE,ALEGRE,2026-05-26 17:01:21.207,ALE05,743CF00000078,...,NaN,NaN,NaN,-20.911188,-41.543312,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Corte de palmitos.,1.0,-1,Até 1 ano
9,Planejamento,303569.0,NaN,CENTRO,DEST-SE,CIVIT,JACARAÍPE,2026-05-26 16:41:53.040,JAC03,707BF00004239,...,NaN,NaN,NaN,-20.148183,-40.193021,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Cortar cipó poste concreto rede MT cabo nu 3x0...,1.0,-1,Até 1 ano
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28528,Planejamento,188165.0,15553520.0,CACHOEIRO,DESC-CI,CACHOEIRO,CACHOEIRO,2024-05-27 17:16:34.000,CIT10,708BF00280715,...,NaN,NaN,Pendente,-20.837512,-41.116855,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Poda de árvores próxima a MT encostando com aç...,1.0,728,Até 2 anos
28529,Planejamento,188163.0,15537518.0,CACHOEIRO,DESC-CI,CACHOEIRO,CACHOEIRO,2024-05-27 17:13:07.000,CIT10,708BF00280715,...,NaN,NaN,Pendente,-20.837092,-41.116236,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Poda de árvores próximas a MT encostando com a...,1.0,728,Até 2 anos
28530,Planejamento,188157.0,10000000.0,CACHOEIRO,DESC-CI,CACHOEIRO,CACHOEIRO,2024-05-27 16:48:37.370,CIT10,708BF00000067,...,NaN,NaN,NaN,-20.838707,-41.115517,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Substituir cruetas da estrutura M2 totalmente ...,1.0,728,Até 2 anos
28531,Planejamento,188154.0,15345178.0,CACHOEIRO,DESC-CI,CACHOEIRO,CACHOEIRO,2024-05-27 16:25:22.400,CIT10,708RL00411556,...,NaN,NaN,Pendente,-20.838332,-41.116478,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Poda de árvore entrelaçada na MT utilizar cami...,1.0,728,Até 2 anos


# Le Hierarquia

In [67]:
#tabela é tabela hierarquia do sql ver posibilidade de pegar do sql 

In [68]:
hierarquia = pd.read_excel(
    fr'{caminho_aux}\hierarquia.xlsx',
)

display(hierarquia)

,EMPRESA,REGIAO,CJ_SIT,CJ_SIGLA,CJ_NOME,SE_SIGLA,SE_NOME,CSD,CIRCUITO,SUPERVISAO
0,380,SUL,401,AFC,AFONSO CLAUDIO,AFC,AFONSO CLAUDIO,CACHOEIRO,AFC01,DESC-VN
1,380,SUL,401,AFC,AFONSO CLAUDIO,AFC,AFONSO CLAUDIO,CACHOEIRO,AFC02,DESC-VN
2,380,SUL,401,AFC,AFONSO CLAUDIO,AFC,AFONSO CLAUDIO,CACHOEIRO,AFC03,DESC-VN
3,380,SUL,401,AFC,AFONSO CLAUDIO,AFC,AFONSO CLAUDIO,CACHOEIRO,AFC04,DESC-VN
4,380,SUL,401,AFC,AFONSO CLAUDIO,AFC,AFONSO CLAUDIO,CACHOEIRO,AFC06,DESC-VN
...,...,...,...,...,...,...,...,...,...,...
606,380,SUL,626,VRI,VILA RICA,VRI,VILA RICA,CACHOEIRO,VRI04,DESC-CI
607,380,SUL,626,VRI,VILA RICA,VRI,VILA RICA,CACHOEIRO,VRI05,DESC-CI
608,380,SUL,626,VRI,VILA RICA,VRI,VILA RICA,CACHOEIRO,VRI06,DESC-CI
609,380,SUL,626,VRI,VILA RICA,VRI,VILA RICA,CACHOEIRO,VRI07,DESC-CI


# Le capacidade de execução

In [69]:
capacidade = pd.read_excel(
    fr'{caminho_aux}\Capacidade Execução.xlsx', #Precisa ser atulizado
    sheet_name='Capacidade_Total'
)

display(capacidade)

,CSD,Atributo,Valor,Data
0,CACHOEIRO,Janeiro,279,2025-01-01
1,CACHOEIRO,Fevereiro,329,2025-02-01
2,CACHOEIRO,Março,329,2025-03-01
3,CACHOEIRO,Abril,517,2025-04-01
4,CACHOEIRO,Maio,611,2025-05-01
...,...,...,...,...
67,NOVA VENÉCIA,Agosto,312,2025-08-01
68,NOVA VENÉCIA,Setembro,312,2025-09-01
69,NOVA VENÉCIA,Outubro,312,2025-10-01
70,NOVA VENÉCIA,Novembro,223,2025-11-01


In [70]:
Divisão_Supervisão = pd.read_excel(
    fr'{caminho_aux}\Capacidade Execução.xlsx', #Precisa ser atulizado
    sheet_name='Divisão por Supervisão'
)

display(Divisão_Supervisão)

,CSD,Supervisão,% do CSD,% Notas Poda,% Notas Manut.
0,CACHOEIRO,DESC-AL,0.412737,0.572998,0.427002
1,CACHOEIRO,DESC-CI,0.374244,0.723089,0.276911
2,CACHOEIRO,DESC-VN,0.213019,0.675970,0.324030
3,CENTRO,DESG-MF,0.038856,0.821472,0.178528
4,CENTRO,DEST-SE,0.315010,0.643152,0.356848
5,CENTRO,DEST-VI,0.149318,0.608355,0.391645
6,CENTRO,DESU-CR,0.241845,0.624933,0.375067
7,CENTRO,DESU-VV,0.254971,0.623853,0.376147
8,GUARAPARI,DESG-GP,0.374846,0.753701,0.246299
9,GUARAPARI,DESG-MF,0.332922,0.821472,0.178528


In [71]:
# print("⚠️ Aviso: Na celula abaixo selecione os meses correspondetes para novo pacote. Pressione Enter para continuar...")
# input()


In [ ]:
import ipywidgets as widgets
from IPython.display import display

meses = [
    "Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
    "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"
]

titulo = widgets.HTML("<b>Selecione os meses do novo pacote:</b>")
seletor_meses = widgets.SelectMultiple(options=meses, value=(), description='Meses', rows=12)
btn_confirmar = widgets.Button(description="Salvar Meses", button_style="success")
saida = widgets.Output()

meses_selecionados = []

def confirmar_meses(_):
    global meses_selecionados
    meses_selecionados = list(seletor_meses.value)
    with saida:
        saida.clear_output()
        print("Meses selecionados:", meses_selecionados)

btn_confirmar.on_click(confirmar_meses)

display(titulo, seletor_meses, btn_confirmar, saida)


HTML(value='<b>Selecione os meses do novo pacote:</b>')

SelectMultiple(description='Meses', options=('Janeiro', 'Fevereiro', 'Março', 'Abril', 'Maio', 'Junho', 'Julho…

Button(button_style='success', description='Salvar Meses', style=ButtonStyle())

Output()

In [74]:
meses_selecionados = list(seletor_meses.value)

if not meses_selecionados:
    raise ValueError("Selecione pelo menos 1 mês antes de continuar.")

print("Meses selecionados:", meses_selecionados)


Meses selecionados: ['Julho', 'Agosto', 'Setembro']


In [75]:
capacidade = capacidade[capacidade['Atributo'].isin(meses_selecionados)]

In [76]:
capacidade_total = capacidade.groupby('CSD').agg({'Valor': 'sum'}).reset_index()
capacidade_total['META CSD'] = capacidade_total['Valor'] * 0.8
display(capacidade_total)

,CSD,Valor,META CSD
0,CACHOEIRO,2371,1896.8
1,CENTRO,2297,1837.6
2,GUARAPARI,1567,1253.6
3,ITARANA,1150,920.0
4,LINHARES,1602,1281.6
5,NOVA VENÉCIA,936,748.8


In [77]:
Divisão_Supervisão_join = Divisão_Supervisão.merge(capacidade_total, left_on='CSD', right_on='CSD', how='left').drop('Valor', axis=1)

Divisão_Supervisão_join['QTD Supervisão'] = np.ceil(
    Divisão_Supervisão_join['META CSD'] *
    Divisão_Supervisão_join['% do CSD']
).astype(int)

Divisão_Supervisão_join['QTD Notas Poda'] = np.ceil(
    Divisão_Supervisão_join['QTD Supervisão'] *
    Divisão_Supervisão_join['% Notas Poda']
).astype(int)

Divisão_Supervisão_join['QTD Notas Manut'] = np.ceil(
    Divisão_Supervisão_join['QTD Supervisão'] *
    Divisão_Supervisão_join['% Notas Manut.']
).astype(int)

display(Divisão_Supervisão_join)

,CSD,Supervisão,% do CSD,% Notas Poda,% Notas Manut.,META CSD,QTD Supervisão,QTD Notas Poda,QTD Notas Manut
0,CACHOEIRO,DESC-AL,0.412737,0.572998,0.427002,1896.8,783,449,335
1,CACHOEIRO,DESC-CI,0.374244,0.723089,0.276911,1896.8,710,514,197
2,CACHOEIRO,DESC-VN,0.213019,0.675970,0.324030,1896.8,405,274,132
3,CENTRO,DESG-MF,0.038856,0.821472,0.178528,1837.6,72,60,13
4,CENTRO,DEST-SE,0.315010,0.643152,0.356848,1837.6,579,373,207
5,CENTRO,DEST-VI,0.149318,0.608355,0.391645,1837.6,275,168,108
6,CENTRO,DESU-CR,0.241845,0.624933,0.375067,1837.6,445,279,167
7,CENTRO,DESU-VV,0.254971,0.623853,0.376147,1837.6,469,293,177
8,GUARAPARI,DESG-GP,0.374846,0.753701,0.246299,1253.6,470,355,116
9,GUARAPARI,DESG-MF,0.332922,0.821472,0.178528,1253.6,418,344,75


# Traz colunas de indicadores para tabela de notas

## Traz colunas de CI,CHI e Ocorrencias

In [78]:
Notas

,Solicitante,id_onr,id_sap,CSD,SUPERVISAO,CJ_NOME,SE_NOME,recebido,alimentador,local_instalacao,...,data_fabricacao_poste,postes,Status_SAP,latitude,longitude,Croqui,observacoes,IdsDeolhoRede,tempo de nota,Classificação Tempo
2,CSD,303587.0,NaN,LINHARES,DESL-LI,CANIVETE,JUNCADO,2026-05-26 17:28:22.813,JUN05,770BF00005113,...,NaN,NaN,NaN,-19.118719,-40.086204,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Carro linha viva vem debaixo poste estrutura N1,1.0,-1,Até 1 ano
3,CSD,303585.0,NaN,LINHARES,DESL-LI,LINHARES,LINHARES,2026-05-26 17:24:08.837,LIN16,723BF00005113,...,NaN,NaN,NaN,-19.118830,-40.086135,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Carro linha viva vem debaixo poste estrutura N1,1.0,-1,Até 1 ano
6,CSD,303580.0,NaN,CACHOEIRO,DESC-AL,MIMOSO DO SUL,MIMOSO,2026-05-26 17:13:26.047,MSU01,743CF00000078,...,NaN,NaN,NaN,-20.916632,-41.539976,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Corte de eucalipto ao lado da BF1020,1.0,-1,Até 1 ano
8,CSD,303577.0,NaN,CACHOEIRO,DESC-AL,ALEGRE,ALEGRE,2026-05-26 17:01:21.207,ALE05,743CF00000078,...,NaN,NaN,NaN,-20.911188,-41.543312,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Corte de palmitos.,1.0,-1,Até 1 ano
9,Planejamento,303569.0,NaN,CENTRO,DEST-SE,CIVIT,JACARAÍPE,2026-05-26 16:41:53.040,JAC03,707BF00004239,...,NaN,NaN,NaN,-20.148183,-40.193021,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Cortar cipó poste concreto rede MT cabo nu 3x0...,1.0,-1,Até 1 ano
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28528,Planejamento,188165.0,15553520.0,CACHOEIRO,DESC-CI,CACHOEIRO,CACHOEIRO,2024-05-27 17:16:34.000,CIT10,708BF00280715,...,NaN,NaN,Pendente,-20.837512,-41.116855,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Poda de árvores próxima a MT encostando com aç...,1.0,728,Até 2 anos
28529,Planejamento,188163.0,15537518.0,CACHOEIRO,DESC-CI,CACHOEIRO,CACHOEIRO,2024-05-27 17:13:07.000,CIT10,708BF00280715,...,NaN,NaN,Pendente,-20.837092,-41.116236,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Poda de árvores próximas a MT encostando com a...,1.0,728,Até 2 anos
28530,Planejamento,188157.0,10000000.0,CACHOEIRO,DESC-CI,CACHOEIRO,CACHOEIRO,2024-05-27 16:48:37.370,CIT10,708BF00000067,...,NaN,NaN,NaN,-20.838707,-41.115517,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Substituir cruetas da estrutura M2 totalmente ...,1.0,728,Até 2 anos
28531,Planejamento,188154.0,15345178.0,CACHOEIRO,DESC-CI,CACHOEIRO,CACHOEIRO,2024-05-27 16:25:22.400,CIT10,708RL00411556,...,NaN,NaN,Pendente,-20.838332,-41.116478,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Poda de árvore entrelaçada na MT utilizar cami...,1.0,728,Até 2 anos


In [79]:
#padroniza coluna de proteção final
Notas['Protecao_Final'] = np.where(
    Notas['dispositivo_protecao'].isna() | 
    (Notas['dispositivo_protecao'].astype(str).str.len() < 3),
    Notas['local_instalacao']+'9',
    Notas['dispositivo_protecao']
)

In [80]:
Notas_Indicadores = Notas.merge(Dim_Ocorrencias, left_on='Protecao_Final', right_on='EQ_INTERRUPCAO', how='left').drop('EQ_INTERRUPCAO', axis=1)
Notas_Indicadores

,Solicitante,id_onr,id_sap,CSD,SUPERVISAO,CJ_NOME,SE_NOME,recebido,alimentador,local_instalacao,...,Croqui,observacoes,IdsDeolhoRede,tempo de nota,Classificação Tempo,Protecao_Final,QTD_Ocorrencias,CI,CHI,QTD_Ocorrencias_90d
0,CSD,303587.0,NaN,LINHARES,DESL-LI,CANIVETE,JUNCADO,2026-05-26 17:28:22.813,JUN05,770BF00005113,...,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Carro linha viva vem debaixo poste estrutura N1,1.0,-1,Até 1 ano,770BF000051139,3,3,15.800000,2
1,CSD,303585.0,NaN,LINHARES,DESL-LI,LINHARES,LINHARES,2026-05-26 17:24:08.837,LIN16,723BF00005113,...,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Carro linha viva vem debaixo poste estrutura N1,1.0,-1,Até 1 ano,723BF000051139,<NA>,<NA>,NaN,<NA>
2,CSD,303580.0,NaN,CACHOEIRO,DESC-AL,MIMOSO DO SUL,MIMOSO,2026-05-26 17:13:26.047,MSU01,743CF00000078,...,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Corte de eucalipto ao lado da BF1020,1.0,-1,Até 1 ano,743CF000000789,<NA>,<NA>,NaN,<NA>
3,CSD,303577.0,NaN,CACHOEIRO,DESC-AL,ALEGRE,ALEGRE,2026-05-26 17:01:21.207,ALE05,743CF00000078,...,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Corte de palmitos.,1.0,-1,Até 1 ano,743CF000000789,<NA>,<NA>,NaN,<NA>
4,Planejamento,303569.0,NaN,CENTRO,DEST-SE,CIVIT,JACARAÍPE,2026-05-26 16:41:53.040,JAC03,707BF00004239,...,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Cortar cipó poste concreto rede MT cabo nu 3x0...,1.0,-1,Até 1 ano,707BF000042399,<NA>,<NA>,NaN,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22621,Planejamento,188165.0,15553520.0,CACHOEIRO,DESC-CI,CACHOEIRO,CACHOEIRO,2024-05-27 17:16:34.000,CIT10,708BF00280715,...,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Poda de árvores próxima a MT encostando com aç...,1.0,728,Até 2 anos,708BF002807159,<NA>,<NA>,NaN,<NA>
22622,Planejamento,188163.0,15537518.0,CACHOEIRO,DESC-CI,CACHOEIRO,CACHOEIRO,2024-05-27 17:13:07.000,CIT10,708BF00280715,...,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Poda de árvores próximas a MT encostando com a...,1.0,728,Até 2 anos,708BF002807159,<NA>,<NA>,NaN,<NA>
22623,Planejamento,188157.0,10000000.0,CACHOEIRO,DESC-CI,CACHOEIRO,CACHOEIRO,2024-05-27 16:48:37.370,CIT10,708BF00000067,...,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Substituir cruetas da estrutura M2 totalmente ...,1.0,728,Até 2 anos,708BF000000679,<NA>,<NA>,NaN,<NA>
22624,Planejamento,188154.0,15345178.0,CACHOEIRO,DESC-CI,CACHOEIRO,CACHOEIRO,2024-05-27 16:25:22.400,CIT10,708RL00411556,...,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Poda de árvore entrelaçada na MT utilizar cami...,1.0,728,Até 2 anos,708RL004115569,<NA>,<NA>,NaN,<NA>


In [81]:
#subistitui valores null por 0
Notas_Indicadores = Notas_Indicadores.fillna({'QTD_Ocorrencias': 0, 'CI': 0, 'CHI': 0, 'QTD_Ocorrencias_90d': 0})
# Convertendo a coluna para inteiro
Notas_Indicadores['QTD_Ocorrencias'] = Notas_Indicadores['QTD_Ocorrencias'].astype('int64')
Notas_Indicadores['CI'] = Notas_Indicadores['CI'].astype('int64')
Notas_Indicadores['QTD_Ocorrencias_90d'] = Notas_Indicadores['QTD_Ocorrencias_90d'].astype('int64')
# Convertendo a coluna para decimal
Notas_Indicadores['CHI'] = Notas_Indicadores['CHI'].astype('float64')

Notas_Indicadores

,Solicitante,id_onr,id_sap,CSD,SUPERVISAO,CJ_NOME,SE_NOME,recebido,alimentador,local_instalacao,...,Croqui,observacoes,IdsDeolhoRede,tempo de nota,Classificação Tempo,Protecao_Final,QTD_Ocorrencias,CI,CHI,QTD_Ocorrencias_90d
0,CSD,303587.0,NaN,LINHARES,DESL-LI,CANIVETE,JUNCADO,2026-05-26 17:28:22.813,JUN05,770BF00005113,...,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Carro linha viva vem debaixo poste estrutura N1,1.0,-1,Até 1 ano,770BF000051139,3,3,15.800000,2
1,CSD,303585.0,NaN,LINHARES,DESL-LI,LINHARES,LINHARES,2026-05-26 17:24:08.837,LIN16,723BF00005113,...,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Carro linha viva vem debaixo poste estrutura N1,1.0,-1,Até 1 ano,723BF000051139,0,0,0.000000,0
2,CSD,303580.0,NaN,CACHOEIRO,DESC-AL,MIMOSO DO SUL,MIMOSO,2026-05-26 17:13:26.047,MSU01,743CF00000078,...,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Corte de eucalipto ao lado da BF1020,1.0,-1,Até 1 ano,743CF000000789,0,0,0.000000,0
3,CSD,303577.0,NaN,CACHOEIRO,DESC-AL,ALEGRE,ALEGRE,2026-05-26 17:01:21.207,ALE05,743CF00000078,...,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Corte de palmitos.,1.0,-1,Até 1 ano,743CF000000789,0,0,0.000000,0
4,Planejamento,303569.0,NaN,CENTRO,DEST-SE,CIVIT,JACARAÍPE,2026-05-26 16:41:53.040,JAC03,707BF00004239,...,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Cortar cipó poste concreto rede MT cabo nu 3x0...,1.0,-1,Até 1 ano,707BF000042399,0,0,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22621,Planejamento,188165.0,15553520.0,CACHOEIRO,DESC-CI,CACHOEIRO,CACHOEIRO,2024-05-27 17:16:34.000,CIT10,708BF00280715,...,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Poda de árvores próxima a MT encostando com aç...,1.0,728,Até 2 anos,708BF002807159,0,0,0.000000,0
22622,Planejamento,188163.0,15537518.0,CACHOEIRO,DESC-CI,CACHOEIRO,CACHOEIRO,2024-05-27 17:13:07.000,CIT10,708BF00280715,...,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Poda de árvores próximas a MT encostando com a...,1.0,728,Até 2 anos,708BF002807159,0,0,0.000000,0
22623,Planejamento,188157.0,10000000.0,CACHOEIRO,DESC-CI,CACHOEIRO,CACHOEIRO,2024-05-27 16:48:37.370,CIT10,708BF00000067,...,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Substituir cruetas da estrutura M2 totalmente ...,1.0,728,Até 2 anos,708BF000000679,0,0,0.000000,0
22624,Planejamento,188154.0,15345178.0,CACHOEIRO,DESC-CI,CACHOEIRO,CACHOEIRO,2024-05-27 16:25:22.400,CIT10,708RL00411556,...,https://coffee.edp.gpti.com.br/olhonarede/pdf/...,Poda de árvore entrelaçada na MT utilizar cami...,1.0,728,Até 2 anos,708RL004115569,0,0,0.000000,0


## Cria coluna de Peso do conjunto

In [82]:
#Versão com priorização por DEC conjunto

# Indicadores = pd.read_csv(
#     fr'{caminho_base}Base_Indicadores_Movel_ES.csv',
#     sep=";",   # tenta ; em vez de ,
#     encoding="utf-8"  # se der erro de acento, usar "latin1"
# )

# display(Indicadores)

# Convertendo a coluna para decimal

# Indicadores['DELTA_DEC'] = (
#     Indicadores['DELTA_DEC']
#         .astype(str)
#         .str.replace(',', '.', regex=False)
#         .str.strip()
# )

# Indicadores['DELTA_DEC'] = pd.to_numeric(
#     Indicadores['DELTA_DEC'],
#     errors='coerce'
# ).fillna(0)

# # Pegando mínimo e máximo
# min_val = Indicadores['DELTA_DEC'].min()
# max_val = Indicadores['DELTA_DEC'].max()

# # Criando nova coluna com pesos
# if max_val == min_val:
#     Indicadores['Peso_Conj'] = 1
# else:
#     Indicadores['Peso_Conj'] = 1 + ((Indicadores['DELTA_DEC'] - min_val) / (max_val - min_val)) * 0.5 # Peso base ajustar se necessario 

# Indicadores

In [83]:
Prioridade_CJ = pd.read_excel(
    fr'{caminho_base}\Prioridade_Conjuntos_ENG.xlsx'
)

display(Prioridade_CJ)

,CSD,SUPERVISAO,CJ_NOME,Delta indicador do conjunto,PRIORIDADE
0,ITARANA,DESI-IT,SANTA MARIA DE JETIBÁ,3.857844,3
1,ITARANA,DESI-JN,JOAO NEIVA,2.902390,3
2,ITARANA,DESI-JN,ARACRUZ,1.146794,1
3,GUARAPARI,DESG-MF,MARECHAL FLORIANO,0.045688,1
4,GUARAPARI,DESG-MT,PIUMA,-0.060480,1
5,ITARANA,DESI-IT,ITARANA,-0.117692,1
6,LINHARES,DESL-SM,SAO MATEUS,-0.302884,1
7,LINHARES,DESL-LI,CANIVETE,-0.322986,1
8,LINHARES,DESI-JN,LINHARES,-0.350952,1
9,LINHARES,DESL-LI,LINHARES,-0.350952,1


In [84]:
# Cria Peso_Conj com base na prioridade
# Regra:
# prioridade = 1 -> 2.0
# prioridade = 2 -> 1.3
# prioridade = 3 -> 1.0


# Normaliza para número (aceita texto como '1', '2', '3')
prioridade_num = pd.to_numeric(Prioridade_CJ['PRIORIDADE'], errors="coerce")

map_peso = {
    1: 2.0,
    2: 1.3,
    3: 1.0
}
Indicadores["Peso_Conj"] = prioridade_num.map(map_peso).fillna(1.0)
Indicadores


,CONJUNTO_DESC,DEC_CONJUNTO,FEC_CONJUNTO,LIM_DEC,LIM_FEC,DELTA_DEC,DELTA_FEC,Peso_Conj
0,SANTA MARIA DE JETIBÁ,"19,973973878042433","7,8231178674247035","17,0","10,0","2,9739738780424325","-2,1768821325752965",1.0
1,JOAO NEIVA,"12,938601960750997","5,0517593765082704","11,0","7,0","1,9386019607509972","-1,9482406234917296",1.0
2,ARACRUZ,"10,034622887655523","4,030288123510358","10,0","7,0","0,034622887655523016","-2,969711876489642",2.0
3,LINHARES,"9,86276105311263","4,017429046617595","10,0","6,0","-0,13723894688737026","-1,982570953382405",2.0
4,SAO MATEUS,"8,806356811068262","4,186173146503875","9,0","7,0","-0,1936431889317376","-2,813826853496125",2.0
5,CANIVETE,"10,615768896672137","6,076698889609774","11,0","7,0","-0,38423110332786337","-0,9233011103902262",2.0
6,GUARAPARI,"7,381296267928971","2,879017682533889","8,0","5,0","-0,6187037320710287","-2,120982317466111",2.0
7,CEASA,"5,288598050825166","2,442470713501029","6,0","5,0","-0,7114019491748342","-2,557529286498971",2.0
8,PIUMA,"7,27313309704376","3,078683136184365","8,0","6,0","-0,7268669029562398","-2,921316863815635",2.0
9,IBES,"4,105852240016425","2,1018503505425583","5,0","4,0","-0,894147759983575","-1,8981496494574417",2.0


In [85]:
Notas_Indicadores = Notas_Indicadores.merge(Indicadores[['CONJUNTO_DESC','Peso_Conj']],left_on='CJ_NOME', right_on='CONJUNTO_DESC', how='left').drop('CONJUNTO_DESC', axis=1)
Notas_Indicadores

,Solicitante,id_onr,id_sap,CSD,SUPERVISAO,CJ_NOME,SE_NOME,recebido,alimentador,local_instalacao,...,observacoes,IdsDeolhoRede,tempo de nota,Classificação Tempo,Protecao_Final,QTD_Ocorrencias,CI,CHI,QTD_Ocorrencias_90d,Peso_Conj
0,CSD,303587.0,NaN,LINHARES,DESL-LI,CANIVETE,JUNCADO,2026-05-26 17:28:22.813,JUN05,770BF00005113,...,Carro linha viva vem debaixo poste estrutura N1,1.0,-1,Até 1 ano,770BF000051139,3,3,15.800000,2,2.0
1,CSD,303585.0,NaN,LINHARES,DESL-LI,LINHARES,LINHARES,2026-05-26 17:24:08.837,LIN16,723BF00005113,...,Carro linha viva vem debaixo poste estrutura N1,1.0,-1,Até 1 ano,723BF000051139,0,0,0.000000,0,2.0
2,CSD,303580.0,NaN,CACHOEIRO,DESC-AL,MIMOSO DO SUL,MIMOSO,2026-05-26 17:13:26.047,MSU01,743CF00000078,...,Corte de eucalipto ao lado da BF1020,1.0,-1,Até 1 ano,743CF000000789,0,0,0.000000,0,1.0
3,CSD,303577.0,NaN,CACHOEIRO,DESC-AL,ALEGRE,ALEGRE,2026-05-26 17:01:21.207,ALE05,743CF00000078,...,Corte de palmitos.,1.0,-1,Até 1 ano,743CF000000789,0,0,0.000000,0,2.0
4,Planejamento,303569.0,NaN,CENTRO,DEST-SE,CIVIT,JACARAÍPE,2026-05-26 16:41:53.040,JAC03,707BF00004239,...,Cortar cipó poste concreto rede MT cabo nu 3x0...,1.0,-1,Até 1 ano,707BF000042399,0,0,0.000000,0,1.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22621,Planejamento,188165.0,15553520.0,CACHOEIRO,DESC-CI,CACHOEIRO,CACHOEIRO,2024-05-27 17:16:34.000,CIT10,708BF00280715,...,Poda de árvores próxima a MT encostando com aç...,1.0,728,Até 2 anos,708BF002807159,0,0,0.000000,0,1.0
22622,Planejamento,188163.0,15537518.0,CACHOEIRO,DESC-CI,CACHOEIRO,CACHOEIRO,2024-05-27 17:13:07.000,CIT10,708BF00280715,...,Poda de árvores próximas a MT encostando com a...,1.0,728,Até 2 anos,708BF002807159,0,0,0.000000,0,1.0
22623,Planejamento,188157.0,10000000.0,CACHOEIRO,DESC-CI,CACHOEIRO,CACHOEIRO,2024-05-27 16:48:37.370,CIT10,708BF00000067,...,Substituir cruetas da estrutura M2 totalmente ...,1.0,728,Até 2 anos,708BF000000679,0,0,0.000000,0,1.0
22624,Planejamento,188154.0,15345178.0,CACHOEIRO,DESC-CI,CACHOEIRO,CACHOEIRO,2024-05-27 16:25:22.400,CIT10,708RL00411556,...,Poda de árvore entrelaçada na MT utilizar cami...,1.0,728,Até 2 anos,708RL004115569,0,0,0.000000,0,1.0


# Cria Coluna de Cluster Final

In [ ]:
# =========================================================
# 1) TRATAMENTO E PADRONIZAÇÃO DOS DADOS
# =========================================================

# Converte colunas numéricas para float
# errors='coerce' transforma valores inválidos em NaN
Notas_Indicadores["CI"] = pd.to_numeric(
    Notas_Indicadores["CI"],
    errors="coerce"
)

Notas_Indicadores["clientes_bloco_protecao"] = pd.to_numeric(
    Notas_Indicadores["Clientes_bloco_protecao"],
    errors="coerce"
).fillna(0)

Notas_Indicadores["Peso_Conj"] = pd.to_numeric(
    Notas_Indicadores["Peso_Conj"],
    errors="coerce"
).fillna(1)

# Garante que a coluna seja string
Notas_Indicadores["Protecao_Final"] = (
    Notas_Indicadores["Protecao_Final"]
    .fillna("")
    .astype(str)
)

# =========================================================
# 2) FATOR DE CLUSTER
# =========================================================

# Peso de prioridade por cluster
cluster_factor = {
    "Muito Alta": 2.0,
    "Alta": 1.7,
    "Média": 1.3,
    "Baixa": 1.0,
    "Muito Baixa": 1.0
}

# Aplica fator do cluster
Notas_Indicadores["Fator_Cluster"] = (
    Notas_Indicadores["Cluster_Priorizacao"]
    .map(cluster_factor)
    .fillna(1.0)
    .astype(float)
)

# =========================================================
# 3) FATOR DE OCORRÊNCIA 90 DIAS
# =========================================================

# Se teve ocorrência nos últimos 90 dias:
# aplica fator de aumento de prioridade
Notas_Indicadores["Fator_Ocorr90d"] = np.where(
    Notas_Indicadores["QTD_Ocorrencias_90d"].fillna(0) > 0,
    1.5,   # peso para reincidência
    1.0
).astype(float)

# =========================================================
# 4) IDENTIFICAÇÃO DE PROTEÇÕES RL E DJ
# =========================================================

# Exemplo:
# 045RL000270969
#    ^^
#    posições 4 e 5 = RL

# Em Python e qualquer outra linguagem de programação:
# O índice começa em 0
#
# 045RL000270969
# 0123456789...
#
# Portanto:
# str[3:5] => "RL"
# o nome desta técnica é string slicing onde pegamos um indice inicial de uma string
# e um indice final, os separamos pelos ":"

mask_protecao = (
    Notas_Indicadores["Protecao_Final"]
    .str[3:5]
    .isin(["RL", "DJ","CF"])
)

# =========================================================
# 5) AJUSTE DO CI
# =========================================================

# Regra:
# Se:
# - CI for nulo OU igual a 0
# - E proteção for RL ou DJ
#
# Então:
# usar clientes_bloco_protecao no lugar do CI

mask_substituir_ci = (
    (
        Notas_Indicadores["CI"].isna()
        | (Notas_Indicadores["CI"] == 0)
    )
    & mask_protecao
)

# Cria CI ajustado
ci_ajustado = (
    Notas_Indicadores["CI"]
    .mask(
        mask_substituir_ci,
        Notas_Indicadores["clientes_bloco_protecao"]
    )
    .fillna(0)
    .astype(float)
)

# =========================================================
# 6) CÁLCULO DA PRIORIDADE FINAL
# =========================================================

Notas_Indicadores["Prioridade_Final"] = (
    ci_ajustado
    * Notas_Indicadores["Fator_Cluster"]
    * Notas_Indicadores["Fator_Ocorr90d"]
    * Notas_Indicadores["Peso_Conj"]
)

# 6.1) Soma prioridade para os 34,5
Notas_Indicadores["Prioridade_Final"] = (
    Notas_Indicadores["Prioridade_Final"]
    + np.where(
        Notas_Indicadores["alimentador"]
        .astype(str)
        .str.contains("34", na=False),
        50000,
        0
    )
)

# =========================================================
# 7) RANKING DE PRIORIDADE
# =========================================================

# Quanto maior a prioridade_final,
# maior a prioridade operacional

Notas_Indicadores["Rank_Prioridade"] = (
    Notas_Indicadores["Prioridade_Final"]
    .rank(method="dense", ascending=False)
    .astype(int)
)


# =========================================================
# 8) ORDENAÇÃO FINAL
# =========================================================

Notas_Indicadores = (
    Notas_Indicadores
    .sort_values(
        by="Prioridade_Final",
        ascending=False
    )
)

# Exibe resultado
Notas_Indicadores

,Solicitante,id_onr,id_sap,CSD,SUPERVISAO,CJ_NOME,SE_NOME,recebido,alimentador,local_instalacao,...,QTD_Ocorrencias,CI,CHI,QTD_Ocorrencias_90d,Peso_Conj,clientes_bloco_protecao,Fator_Cluster,Fator_Ocorr90d,Prioridade_Final,Rank_Prioridade
13031,Planejamento,236127.0,16317571.0,ITARANA,DESI-JN,ARACRUZ,ARACRUZ,2025-06-12 18:27:17.983,ARA04,714CF00001005,...,10,25499,30191.033333,1,2.0,4186.0,1.7,1.5,130044.9,1
13048,Planejamento,236091.0,16317351.0,ITARANA,DESI-JN,ARACRUZ,ARACRUZ,2025-06-12 18:27:09.030,ARA04,714CF00001007,...,10,25499,30191.033333,1,2.0,4186.0,1.7,1.5,130044.9,1
15335,Planejamento,226908.0,16137187.0,CENTRO,DESU-CR,CARIACICA,CARIACICA,2025-03-28 16:43:08.530,CRC14,703FF00001614,...,4,28107,2911.733333,0,2.0,5529.0,1.7,1.0,95563.8,2
15340,Planejamento,226899.0,16137190.0,CENTRO,DESU-CR,CARIACICA,CARIACICA,2025-03-28 16:43:07.713,CRC14,703FF00001614,...,4,28107,2911.733333,0,2.0,5529.0,1.7,1.0,95563.8,2
6759,Planejamento,260653.0,16861019.0,CENTRO,DESU-CR,CARIACICA,CARIACICA,2026-01-06 14:39:56.043,CRC14,703CF00314573,...,4,28107,2911.733333,0,2.0,5529.0,1.7,1.0,95563.8,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9472,Planejamento,249496.0,10000000.0,CENTRO,DESU-CR,CEASA,CEASA,2025-09-24 09:45:03.303,CEA03,704BF00310685,...,0,0,0.000000,0,2.0,610.0,1.0,1.0,0.0,3781
12928,Planejamento,236473.0,16323954.0,LINHARES,DESL-LI,LINHARES,LINHARES,2025-06-16 20:59:04.607,LIN10,723CF04077279,...,0,0,0.000000,0,2.0,0.0,1.7,1.0,0.0,3781
22554,Planejamento,188419.0,15344968.0,ITARANA,DESI-IT,ITARANA,ITARANA,2024-05-28 22:25:40.080,ITR04,721BF00000238,...,0,0,0.000000,0,2.0,139.0,1.7,1.0,0.0,3781
22553,Planejamento,188420.0,15344967.0,ITARANA,DESI-IT,ITARANA,ITARANA,2024-05-28 22:25:40.250,ITR04,721BF00000238,...,0,0,0.000000,0,2.0,139.0,1.7,1.0,0.0,3781


In [87]:
#salva planinha no computador 
#Notas_Indicadores.to_excel(fr'{caminho_base}Notas_Rank.xlsx',index=False)

# Seleção das Notas

In [88]:
##Faz seleeção das notas sem agruapar por equipamento não esta seno usado

# Notas_Indicadores["Notas Selecionadas"] = 0
# Notas_Indicadores["Tipo Nota Selecionada"] = pd.NA
# Notas_Indicadores["Etapa Selecionada"] = pd.NA

# faltas = []
# resumo_etapas = []

# def selecionar_em_etapas(df, base_mask, qtd_alvo, tipo_nota, csd, sup):
#     faltante = int(max(0, qtd_alvo))
#     selecionados_total = []

#     # contadores por etapa
#     etapas_info = [
#         ("1 Planejamento + Até 1 ano", "Planejamento", "Até 1 ano"),
#         ("2 Planejamento + Até 2 anos", "Planejamento", "Até 2 anos"),
#         ("3 CSD + Até 1 ano", "CSD", "Até 1 ano"),
#         ("4 CSD + Até 2 anos", "CSD", "Até 2 anos"),
#     ]
#     cont_etapas = {e[0]: 0 for e in etapas_info}

#     for nome_etapa, solicitante, classe_tempo in etapas_info:
#         if faltante <= 0:
#             break

#         mask = (
#             base_mask
#             & (df["Notas Selecionadas"] == 0)
#             & (df["Solicitante"] == solicitante)
#             & (df["Classificação Tempo"].isin([classe_tempo, "Até 2 ano"] if classe_tempo == "Até 2 anos" else [classe_tempo]))
#         )

#         cand = df.loc[mask].sort_values(
#             ["Rank_Prioridade", "Priorizacao"],
#             ascending=[True, False]
#         )

#         idx = cand.head(faltante).index
#         qtd_sel = len(idx)

#         if qtd_sel > 0:
#             df.loc[idx, "Notas Selecionadas"] = 1
#             df.loc[idx, "Tipo Nota Selecionada"] = tipo_nota
#             df.loc[idx, "Etapa Selecionada"] = nome_etapa
#             selecionados_total.extend(idx.tolist())
#             cont_etapas[nome_etapa] += qtd_sel
#             faltante -= qtd_sel

#     # resumo por CSD/Supervisão/Tipo
#     resumo_etapas.append({
#         "CSD": csd,
#         "Supervisão": sup,
#         "Tipo": tipo_nota,
#         "Meta": int(qtd_alvo),
#         "Selecionadas": int(qtd_alvo - faltante),
#         "1 Planejamento + Até 1 ano": cont_etapas["1 Planejamento + Até 1 ano"],
#         "2 Planejamento + Até 2 anos": cont_etapas["2 Planejamento + Até 2 anos"],
#         "3 CSD + Até 1 ano": cont_etapas["3 CSD + Até 1 ano"],
#         "4 CSD + Até 2 anos": cont_etapas["4 CSD + Até 2 anos"],
#         "Faltante": int(faltante),
#     })

#     return faltante, selecionados_total


# for _, r in Divisão_Supervisão_join.iterrows():
#     csd = r["CSD"]
#     sup = r["Supervisão"]

#     base_csd_sup = (
#         (Notas_Indicadores["CSD"] == csd) &
#         (Notas_Indicadores["SUPERVISAO"] == sup)
#     )

#     # PODA
#     qtd_poda = int(np.nan_to_num(r["QTD Notas Poda"], nan=0))
#     base_poda = base_csd_sup & Notas_Indicadores["descrição_conjunto"].str.contains("Poda", case=False, na=False)

#     faltante_poda, _ = selecionar_em_etapas(
#         Notas_Indicadores, base_poda, qtd_poda, "Poda", csd, sup
#     )

#     if faltante_poda > 0:
#         faltas.append({
#             "CSD": csd,
#             "Supervisão": sup,
#             "Tipo": "Poda",
#             "Quantidade Faltante": faltante_poda
#         })

#     # MANUT
#     qtd_manut = int(np.nan_to_num(r["QTD Notas Manut"], nan=0))
#     base_manut = base_csd_sup & Notas_Indicadores["descrição_conjunto"].str.contains("Manut", case=False, na=False)

#     faltante_manut, _ = selecionar_em_etapas(
#         Notas_Indicadores, base_manut, qtd_manut, "Manut", csd, sup
#     )

#     if faltante_manut > 0:
#         faltas.append({
#             "CSD": csd,
#             "Supervisão": sup,
#             "Tipo": "Manut",
#             "Quantidade Faltante": faltante_manut
#         })

# # =========================================================
# # DataFrames finais
# # =========================================================
# faltas_df = pd.DataFrame(faltas)
# resumo_etapas_df = pd.DataFrame(resumo_etapas)

# print("Resumo por CSD/Supervisão/Tipo:")
# display(resumo_etapas_df.sort_values(["CSD", "Supervisão", "Tipo"]))

# if faltas_df.empty:
#     print("Sem faltas.")
# else:
#     print("Faltas encontradas:")
#     display(faltas_df.sort_values(["CSD", "Supervisão", "Tipo"]))


In [89]:
# Faz seleção das notas pelo ranking ja definido e etapas de filtros
# Agrupar notas de mesmo bloco de manutenção do planejamento
Notas_Indicadores["Notas Selecionadas"] = 0
Notas_Indicadores["Tipo Nota Selecionada"] = pd.NA
Notas_Indicadores["Etapa Selecionada"] = pd.NA

faltas = []
resumo_etapas = []

def selecionar_em_etapas(df, base_mask, qtd_alvo, tipo_nota, csd, sup, agrupar_por_protecao=False):
    faltante = int(max(0, qtd_alvo))
    selecionados_total = []

    etapas_info = [
        ("1 Planejamento + Até 1 ano", "Planejamento", "Até 1 ano"),
        ("2 Planejamento + Até 2 anos", "Planejamento", "Até 2 anos"),
        ("3 CSD + Até 1 ano", "CSD", "Até 1 ano"),
        ("4 CSD + Até 2 anos", "CSD", "Até 2 anos"),
    ]
    cont_etapas = {e[0]: 0 for e in etapas_info}

    for nome_etapa, solicitante, classe_tempo in etapas_info:
        if faltante <= 0:
            break

        classes_tempo = [classe_tempo, "Até 2 ano"] if classe_tempo == "Até 2 anos" else [classe_tempo]

        mask = (
            base_mask
            & (df["Notas Selecionadas"] == 0)
            & (df["Solicitante"] == solicitante)
            & (df["Classificação Tempo"].isin(classes_tempo))
        )

        cand = df.loc[mask].sort_values(
            ["Rank_Prioridade", "Priorizacao"],
            ascending=[True, False]
        )

        # if cand.empty:
        #     continue

        if not agrupar_por_protecao:
            idx_sel = cand.head(faltante).index.tolist()
        else:
            # Manut: percorre equipamentos por ordem de prioridade
            # e seleciona o bloco do mesmo Protecao_Final antes de ir ao próximo.
            cand = cand.copy()
            cand["_ordem"] = np.arange(len(cand))
            cand["_prot_key"] = np.where(
                cand["Protecao_Final"].notna(),
                cand["Protecao_Final"].astype(str),
                "__SEM_PROTECAO__" + cand.index.astype(str)
            )

            grupos = (
                cand.groupby("_prot_key", sort=False)
                .agg(
                    primeira_ordem=("_ordem", "min"),
                    idx=("Rank_Prioridade", lambda s: s.index.tolist())
                )
                .sort_values("primeira_ordem")
            )

            idx_sel = []
            for _, g in grupos.iterrows():
                if faltante <= 0:
                    break

                idx_grupo = g["idx"]
                if len(idx_grupo) <= faltante:
                    idx_add = idx_grupo
                else:
                    idx_add = idx_grupo[:faltante]  # até preencher

                idx_sel.extend(idx_add)
                faltante -= len(idx_add)

        qtd_sel = len(idx_sel)
        if qtd_sel > 0:
            df.loc[idx_sel, "Notas Selecionadas"] = 1
            df.loc[idx_sel, "Tipo Nota Selecionada"] = tipo_nota
            df.loc[idx_sel, "Etapa Selecionada"] = nome_etapa
            selecionados_total.extend(idx_sel)
            cont_etapas[nome_etapa] += qtd_sel  

            # No fluxo sem agrupamento, faltante ainda não foi abatido
            if not agrupar_por_protecao:
                faltante -= qtd_sel

    resumo_etapas.append({
        "CSD": csd,
        "Supervisão": sup,
        "Tipo": tipo_nota,
        "Meta": int(qtd_alvo),
        "Selecionadas": int(qtd_alvo - faltante),
        "1 Planejamento + Até 1 ano": cont_etapas["1 Planejamento + Até 1 ano"],
        "2 Planejamento + Até 2 anos": cont_etapas["2 Planejamento + Até 2 anos"],
        "3 CSD + Até 1 ano": cont_etapas["3 CSD + Até 1 ano"],
        "4 CSD + Até 2 anos": cont_etapas["4 CSD + Até 2 anos"],
        "Faltante": int(faltante),
    })

    return faltante, selecionados_total


for _, r in Divisão_Supervisão_join.iterrows(): 
    csd = r["CSD"]
    sup = r["Supervisão"]

    base_csd_sup = (
        (Notas_Indicadores["CSD"] == csd) &
        (Notas_Indicadores["SUPERVISAO"] == sup)
    )

    # PODA
    qtd_poda = int(np.nan_to_num(r["QTD Notas Poda"], nan=0))
    base_poda = base_csd_sup & Notas_Indicadores["descrição_conjunto"].str.contains("Poda", case=False, na=False)

    faltante_poda, _ = selecionar_em_etapas(
        Notas_Indicadores, base_poda, qtd_poda, "Poda", csd, sup, agrupar_por_protecao=False
    )

    if faltante_poda > 0:
        faltas.append({
            "CSD": csd,
            "Supervisão": sup,
            "Tipo": "Poda",
            "Quantidade Faltante": faltante_poda
        })

    # MANUT (agrupando por Protecao_Final)
    qtd_manut = int(np.nan_to_num(r["QTD Notas Manut"], nan=0))
    base_manut = base_csd_sup & Notas_Indicadores["descrição_conjunto"].str.contains("Manut", case=False, na=False)

    faltante_manut, _ = selecionar_em_etapas(
        Notas_Indicadores, base_manut, qtd_manut, "Manut", csd, sup, agrupar_por_protecao=True
    )

    if faltante_manut > 0:
        faltas.append({
            "CSD": csd,
            "Supervisão": sup,
            "Tipo": "Manut",
            "Quantidade Faltante": faltante_manut
        })

# =========================================================
# DataFrames finais
# ========================================================= 
faltas_df = pd.DataFrame(faltas)
resumo_etapas_df = pd.DataFrame(resumo_etapas)

print("Resumo por CSD/Supervisão/Tipo:")
display(resumo_etapas_df.sort_values(["CSD", "Supervisão", "Tipo"]))

if faltas_df.empty:
    print("Sem faltas.")
else:
    print("Faltas encontradas:")
    display(faltas_df.sort_values(["CSD", "Supervisão", "Tipo"]))


Resumo por CSD/Supervisão/Tipo:


,CSD,Supervisão,Tipo,Meta,Selecionadas,1 Planejamento + Até 1 ano,2 Planejamento + Até 2 anos,3 CSD + Até 1 ano,4 CSD + Até 2 anos,Faltante
1,CACHOEIRO,DESC-AL,Manut,335,335,287,48,0,0,0
0,CACHOEIRO,DESC-AL,Poda,449,449,284,115,50,0,0
3,CACHOEIRO,DESC-CI,Manut,197,197,197,0,0,0,0
2,CACHOEIRO,DESC-CI,Poda,514,514,514,0,0,0,0
5,CACHOEIRO,DESC-VN,Manut,132,132,132,0,0,0,0
4,CACHOEIRO,DESC-VN,Poda,274,274,274,0,0,0,0
7,CENTRO,DESG-MF,Manut,13,13,1,12,0,0,0
6,CENTRO,DESG-MF,Poda,60,60,18,42,0,0,0
9,CENTRO,DEST-SE,Manut,207,97,69,28,0,0,110
8,CENTRO,DEST-SE,Poda,373,265,245,19,1,0,108


Faltas encontradas:


,CSD,Supervisão,Tipo,Quantidade Faltante
1,CENTRO,DEST-SE,Manut,110
0,CENTRO,DEST-SE,Poda,108
2,NOVA VENÉCIA,DESN-SF,Poda,39


# Salva tabelas finais

In [90]:
Notas_Indicadores

,Solicitante,id_onr,id_sap,CSD,SUPERVISAO,CJ_NOME,SE_NOME,recebido,alimentador,local_instalacao,...,QTD_Ocorrencias_90d,Peso_Conj,clientes_bloco_protecao,Fator_Cluster,Fator_Ocorr90d,Prioridade_Final,Rank_Prioridade,Notas Selecionadas,Tipo Nota Selecionada,Etapa Selecionada
13031,Planejamento,236127.0,16317571.0,ITARANA,DESI-JN,ARACRUZ,ARACRUZ,2025-06-12 18:27:17.983,ARA04,714CF00001005,...,1,2.0,4186.0,1.7,1.5,130044.9,1,1,Manut,1 Planejamento + Até 1 ano
13048,Planejamento,236091.0,16317351.0,ITARANA,DESI-JN,ARACRUZ,ARACRUZ,2025-06-12 18:27:09.030,ARA04,714CF00001007,...,1,2.0,4186.0,1.7,1.5,130044.9,1,1,Manut,1 Planejamento + Até 1 ano
15335,Planejamento,226908.0,16137187.0,CENTRO,DESU-CR,CARIACICA,CARIACICA,2025-03-28 16:43:08.530,CRC14,703FF00001614,...,0,2.0,5529.0,1.7,1.0,95563.8,2,0,<NA>,<NA>
15340,Planejamento,226899.0,16137190.0,CENTRO,DESU-CR,CARIACICA,CARIACICA,2025-03-28 16:43:07.713,CRC14,703FF00001614,...,0,2.0,5529.0,1.7,1.0,95563.8,2,0,<NA>,<NA>
6759,Planejamento,260653.0,16861019.0,CENTRO,DESU-CR,CARIACICA,CARIACICA,2026-01-06 14:39:56.043,CRC14,703CF00314573,...,0,2.0,5529.0,1.7,1.0,95563.8,2,1,Poda,1 Planejamento + Até 1 ano
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9472,Planejamento,249496.0,10000000.0,CENTRO,DESU-CR,CEASA,CEASA,2025-09-24 09:45:03.303,CEA03,704BF00310685,...,0,2.0,610.0,1.0,1.0,0.0,3781,0,<NA>,<NA>
12928,Planejamento,236473.0,16323954.0,LINHARES,DESL-LI,LINHARES,LINHARES,2025-06-16 20:59:04.607,LIN10,723CF04077279,...,0,2.0,0.0,1.7,1.0,0.0,3781,0,<NA>,<NA>
22554,Planejamento,188419.0,15344968.0,ITARANA,DESI-IT,ITARANA,ITARANA,2024-05-28 22:25:40.080,ITR04,721BF00000238,...,0,2.0,139.0,1.7,1.0,0.0,3781,0,<NA>,<NA>
22553,Planejamento,188420.0,15344967.0,ITARANA,DESI-IT,ITARANA,ITARANA,2024-05-28 22:25:40.250,ITR04,721BF00000238,...,0,2.0,139.0,1.7,1.0,0.0,3781,0,<NA>,<NA>


In [91]:
Notas_Indicadores.to_excel(fr'{caminho_base}\Notas_Selecionadas.xlsx',index=False)

In [92]:
with pd.ExcelWriter(fr"{caminho_base}\Resumo_Selecao_Notas.xlsx", engine="openpyxl") as writer:
    resumo_etapas_df.to_excel(writer, sheet_name="resumo_etapas", index=False)
    faltas_df.to_excel(writer, sheet_name="faltas", index=False)